<a href="https://colab.research.google.com/github/andilMc/gemmafro-e2b/blob/main/notebooks/00_setup_environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 — Setup environnement (Phase 0)

Notebook à ouvrir directement dans **Google Colab** (GPU T4 gratuit). Correspond aux étapes 3 à 14 de la Phase 0 décrite dans `docs/WORKFLOW.md`.

**Avant de commencer :** menu *Exécution* → *Modifier le type d'exécution* → Accélérateur matériel : **T4 GPU**.

In [ ]:
# Vérifie le GPU alloué par Colab (nom et VRAM).
!nvidia-smi

Fri Aug 14 12:46:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Monte Google Drive : checkpoints et logs y sont sauvegardés pour survivre à la déconnexion de la session.
from google.colab import drive
drive.mount('/content/drive')  # demande l'autorisation d'accès au Drive

Mounted at /content/drive


In [3]:
# Crée l'arborescence sur Drive (checkpoints, logs, submissions). Les données viennent du dépôt Git cloné,
# pas de duplication.
import os

PROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'  # racine du projet sur Drive
for sub in ['checkpoints', 'logs', 'submissions']:  # un dossier par type de sortie
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)  # crée le dossier sans erreur s'il existe déjà

print(os.listdir(PROJECT_DIR))  # vérifie que les dossiers existent

['checkpoints', 'logs', 'submissions']


In [4]:
# Clone le dépôt du projet (data/, docs/, notebooks/), ou le met à jour s'il est déjà présent.
import os

REPO_DIR = '/content/gemmafro-e2b'
if not os.path.isdir(REPO_DIR):  # clone seulement si le dépôt est absent
    # télécharge le dépôt du projet
    !git clone https://github.com/andilMc/gemmafro-e2b.git {REPO_DIR}
else:
    # sinon le met à jour
    !cd {REPO_DIR} && git pull

DATA_DIR = f'{REPO_DIR}/data'  # dossier des CSV
print(os.listdir(DATA_DIR))  # vérifie que Train/Val/Test sont présents

Cloning into '/content/gemmafro-e2b'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 37 (delta 10), reused 29 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 5.91 MiB | 578.00 KiB/s, done.
Resolving deltas: 100% (10/10), done.
['Val.csv', 'Test.csv', 'SampleSubmission.csv', 'Train.csv']


In [9]:
# Installe les dépendances (à refaire à chaque session, Colab n'est pas persistant) :
# transformers/peft/trl/bitsandbytes pour le modèle, LoRA et la quantification 4-bit ; rouge-score pour les
# métriques.
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets rouge-score pandas

In [13]:
# Se connecte à Hugging Face avec le token stocké dans les Colab Secrets (clé HF_TOKEN, accès notebook
# activé).
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))  # authentifie la session avec le token HF_TOKEN

In [14]:
# Test de chargement du modèle en 4-bit (QLoRA).
# ⚠️ Avant de lancer : accepter la licence du modèle sur sa page Hugging Face (Agree and access repository)
# et vérifier le nom exact du checkpoint.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "google/gemma-4-E2B-it"  # ⚠️ à remplacer par le nom exact du checkpoint vérifié sur HF

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # quantifie les poids en 4 bits pour tenir en VRAM
    bnb_4bit_quant_type="nf4",  # format NormalFloat4, adapté aux poids de réseaux de neurones
    bnb_4bit_compute_dtype=torch.float16,   # fp16, pas bf16 : T4 ne le supporte pas efficacement
    bnb_4bit_use_double_quant=True,  # quantifie aussi les constantes de quantification (gain mémoire)
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # tokenizer du modèle
model = AutoModelForCausalLM.from_pretrained(  # charge le modèle quantifié
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",  # répartit les couches sur le GPU automatiquement
)
print(f"Empreinte mémoire : {model.get_memory_footprint() / 1e9:.2f} Go")  # mémoire occupée par le modèle chargé

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Empreinte mémoire : 6.70 Go


In [15]:
# Test rapide de génération : valide que le chat template produit une réponse cohérente.
messages = [{"role": "user", "content": "Bonjour, peux-tu répondre en français ?"}]  # un tour utilisateur
# applique le chat template et ajoute l'amorce de la réponse
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)  # texte → tenseurs, envoyés sur le GPU
output = model.generate(**inputs, max_new_tokens=50)  # génère jusqu'à 50 nouveaux tokens
print(tokenizer.decode(output[0], skip_special_tokens=True))  # tokens → texte lisible

user
Bonjour, peux-tu répondre en français ?
model
Oui, absolument ! Je peux tout à fait répondre en français.

**Comment puis-je vous aider aujourd'hui ?** 😊


In [16]:
# Vérifie que les CSV du dépôt cloné se chargent bien.
import pandas as pd

train = pd.read_csv(f'{DATA_DIR}/Train.csv')  # questions/réponses d'entraînement
val   = pd.read_csv(f'{DATA_DIR}/Val.csv')  # questions/réponses de validation
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')  # questions de test (sans réponse)

print('Train:', train.shape, '| Val:', val.shape, '| Test:', test.shape)  # dimensions (lignes, colonnes)
train.head(3)  # aperçu des 3 premières lignes

Train: (29815, 4) | Val: (6686, 4) | Test: (2618, 3)


,ID,input,output,subset
0,ID_TR_Aka_Gha_A3B1799D,Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a...,Mmabun betumi aboa atipɛnfo a ebia nsa anaa nn...,Aka_Gha
1,ID_TR_Aka_Gha_1C80317F,Edinnsiananmu bɛn na nnipa a ɛsono wɔn bɔbeasu...,"Wɔ Ghana mu no, amanmmra no gye binary gender ...",Aka_Gha
2,ID_TR_Aka_Gha_06671AD1,Ɔkwan bɛn so na ɔbarima ne ɔbea nna a wɔtwe wɔ...,Sɛ wɔtwe wɔn ho fi nna mu anaasɛ wɔtwentwɛn wɔ...,Aka_Gha


---
**Si toutes les cellules ci-dessus s'exécutent sans erreur : l'environnement est prêt.** Étape suivante : Phase 1 — Analyse exploratoire des données (voir `docs/WORKFLOW.md`).